# 03a — Random Forest baseline (Morgan fingerprints)

**Project:** Lightweight domain adaptation of small LLMs for chemistry using LoRA and QLoRA (MSc FYP).

**Phase 3 covers two baselines:**
- **03a (this notebook):** Random Forest on Morgan fingerprints — the *classical* baseline.
- **03b (next notebook):** ChemBERTa-2 fine-tune — the *pretrained-LLM* baseline.

**Why we do baselines first (supervisor: "non-negotiable"):** without these numbers, later LoRA/QLoRA scores are meaningless. "LoRA hits 78% AUC" only matters if we know RF gets 73% and ChemBERTa gets 80%. Baselines anchor every later claim.

**What this notebook does:**
1. Loads the Phase 02 scaffold splits from Drive (no reprocessing)
2. Converts every SMILES to a **Morgan fingerprint** (radius 2, 2048 bits) — the standard cheminformatics featurisation
3. Trains a **Random Forest** per dataset using sklearn defaults
4. Evaluates on the held-out test set with the supervisor's metrics:
   - BBBP & BACE: **ROC-AUC** (primary), **F1** (secondary)
   - ESOL: **RMSE** (primary), **MAE** (secondary)
5. Writes results to `/MyDrive/chemistry-peft-fyp/results/baselines.csv` — the shared scoreboard that 03b will append to.

## 1. Install + imports

RDKit (for Morgan fingerprints), pandas, scikit-learn. All small; ~30 seconds.

In [ ]:
%pip install -q rdkit pandas scikit-learn

In [ ]:
import os, random
import numpy as np
import pandas as pd
from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import roc_auc_score, f1_score, mean_squared_error, mean_absolute_error

RDLogger.DisableLog("rdApp.*")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print("pandas       :", pd.__version__)
import sklearn; print("scikit-learn :", sklearn.__version__)

## 2. Mount Drive + load the Phase 02 splits

Reading the 9 CSVs we saved in `02_dataset_prep`. No re-processing — Phase 03 trusts Phase 02's work.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DATA_DIR    = "/content/drive/MyDrive/chemistry-peft-fyp/data"
RESULTS_DIR = "/content/drive/MyDrive/chemistry-peft-fyp/results"
os.makedirs(RESULTS_DIR, exist_ok=True)

RESULTS_CSV = f"{RESULTS_DIR}/baselines.csv"
print("Data dir    :", DATA_DIR)
print("Results CSV :", RESULTS_CSV)

In [ ]:
def load_splits(name):
    prefix = name.lower()
    tr = pd.read_csv(f"{DATA_DIR}/{prefix}_train.csv")
    va = pd.read_csv(f"{DATA_DIR}/{prefix}_val.csv")
    te = pd.read_csv(f"{DATA_DIR}/{prefix}_test.csv")
    print(f"{name:5s}: train={len(tr)}  val={len(va)}  test={len(te)}")
    return tr, va, te

splits = {name: load_splits(name) for name in ["BBBP", "BACE", "ESOL"]}

## 3. Featurise: SMILES → Morgan fingerprints

**Morgan fingerprints**, radius 2, 2048 bits. Also called **ECFP4** in the literature (radius 2 ↔ ECFP**4**, because the diameter is 4 bonds).

**How they work:** for each atom in the molecule, look at its local neighbourhood up to radius 2 (the atom + its neighbours + neighbours' neighbours). Hash that local environment into a number from 0–2047, and set that bit in the output vector. The final 2048-bit vector encodes which structural features are present.

**Why these are the cheminformatics default:** they're rotation/numbering-invariant, fast to compute, and capture enough local structure that classical ML (Random Forest, SVM) can do real work. They're the standard input to MoleculeNet's reference baselines.

In [ ]:
RADIUS  = 2
N_BITS  = 2048
_GEN    = AllChem.GetMorganGenerator(radius=RADIUS, fpSize=N_BITS)

def smiles_to_fp(smi):
    """Return a 2048-bit Morgan fingerprint as a numpy array of 0/1 ints."""
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return None
    fp = _GEN.GetFingerprintAsNumPy(mol)
    return fp.astype(np.uint8)

def featurise(df):
    """DataFrame with 'smiles', 'label' → (X, y) ready for sklearn."""
    fps = [smiles_to_fp(s) for s in df["smiles"]]
    keep = [i for i, fp in enumerate(fps) if fp is not None]
    if len(keep) < len(fps):
        print(f"  dropped {len(fps) - len(keep)} unparseable rows")
    X = np.stack([fps[i] for i in keep])
    y = df["label"].iloc[keep].to_numpy()
    return X, y

# Smoke test on one row
sample_fp = smiles_to_fp("CC(=O)Oc1ccccc1C(=O)O")  # aspirin
print("Aspirin fingerprint shape:", sample_fp.shape, "bits set:", int(sample_fp.sum()))

## 4. Train Random Forest on each dataset

**Random Forest** = ensemble of decision trees, each trained on a random sample of rows + features. Robust, fast, no tuning required. The cheminformatics default baseline for ~15 years.

**Hyperparameters:** sklearn defaults (`n_estimators=100`, no `max_depth`). Deliberately not tuned — for a baseline, the honest claim is "out-of-the-box scikit-learn." Tuned baselines stop being baselines.

In [ ]:
results = []   # list of dicts → DataFrame at the end

def train_eval_classification(name):
    print(f"\n=== {name} (classification) ===")
    tr, va, te = splits[name]
    print("Featurising train...");  X_tr, y_tr = featurise(tr)
    print("Featurising test ...");  X_te, y_te = featurise(te)

    clf = RandomForestClassifier(n_estimators=100, random_state=SEED, n_jobs=-1)
    clf.fit(X_tr, y_tr)

    proba = clf.predict_proba(X_te)[:, 1]
    pred  = clf.predict(X_te)
    auc = roc_auc_score(y_te, proba)
    f1  = f1_score(y_te, pred)
    print(f"ROC-AUC: {auc:.4f}   F1: {f1:.4f}")

    results.append({
        "model": "RandomForest+Morgan", "dataset": name, "task": "classification",
        "metric_primary": "roc_auc",   "value_primary": round(auc, 4),
        "metric_secondary": "f1",      "value_secondary": round(f1, 4),
        "n_train": len(y_tr), "n_test": len(y_te),
    })

def train_eval_regression(name):
    print(f"\n=== {name} (regression) ===")
    tr, va, te = splits[name]
    print("Featurising train...");  X_tr, y_tr = featurise(tr)
    print("Featurising test ...");  X_te, y_te = featurise(te)

    reg = RandomForestRegressor(n_estimators=100, random_state=SEED, n_jobs=-1)
    reg.fit(X_tr, y_tr)

    pred = reg.predict(X_te)
    rmse = float(np.sqrt(mean_squared_error(y_te, pred)))
    mae  = float(mean_absolute_error(y_te, pred))
    print(f"RMSE: {rmse:.4f}   MAE: {mae:.4f}")

    results.append({
        "model": "RandomForest+Morgan", "dataset": name, "task": "regression",
        "metric_primary": "rmse",      "value_primary": round(rmse, 4),
        "metric_secondary": "mae",     "value_secondary": round(mae, 4),
        "n_train": len(y_tr), "n_test": len(y_te),
    })

train_eval_classification("BBBP")
train_eval_classification("BACE")
train_eval_regression("ESOL")

## 5. Save results to the shared scoreboard

We're writing to `results/baselines.csv` on Drive. Notebook 03b (ChemBERTa) will **append** its rows to the same file. Phase 06 (final comparison) reads from it.

If a previous run wrote RF rows already, we replace them rather than duplicating — so re-running this notebook is idempotent.

In [ ]:
new_rows = pd.DataFrame(results)

if os.path.exists(RESULTS_CSV):
    existing = pd.read_csv(RESULTS_CSV)
    # Drop any previous rows for this model so re-runs replace, not duplicate.
    existing = existing[existing["model"] != "RandomForest+Morgan"]
    combined = pd.concat([existing, new_rows], ignore_index=True)
else:
    combined = new_rows

combined.to_csv(RESULTS_CSV, index=False)
print(f"Wrote {len(new_rows)} rows to {RESULTS_CSV}")
print("\nCurrent scoreboard:")
print(combined.to_string(index=False))

## 6. Done

**Phase 03 checklist (RF half):**
- ✅ Computed Morgan fingerprints (radius 2, 2048 bits) for every molecule
- ✅ Trained Random Forest on BBBP train set
- ✅ Evaluated on BBBP test: ROC-AUC, F1
- ✅ Repeated for BACE (classification) and ESOL (regression — RMSE, MAE)
- ✅ Recorded numbers in `results/baselines.csv`

**Still pending (notebook 03b):**
- ⏳ Load ChemBERTa-2 from Hugging Face
- ⏳ Fine-tune ChemBERTa-2 on BBBP
- ⏳ Evaluate ChemBERTa-2 on BBBP, BACE, ESOL
- ⏳ Append to `results/baselines.csv`

**Expected RF numbers (rough guide, from MoleculeNet literature):**
- BBBP ROC-AUC ≈ 0.70–0.75
- BACE ROC-AUC ≈ 0.78–0.85
- ESOL RMSE   ≈ 1.0–1.3

If your numbers fall well outside these ranges, something's off — flag it in chat and we'll investigate.